# Day 6(M2 Day02) 실습 — LangChain Tool 바인딩과 외부 API 연결

**목표**: 도구를 여러 개 등록하고, AI가 고른 이름으로 알맞은 함수를 실행한다. 외부 API도 도구로 감싼다.
**구성**: Part 1 도구 2개·이름 dispatch(+오류 대응) → Part 2 외부 API·설명의 힘(+회사 리뷰 도구) → Part 3 도구 모음 루프(+M2 Day01 면접 코치 확장)

> **참고:** 실습 전 가상환경 활성화, `.env`의 OpenAI API 키를 확인한다.

## 0. 환경 준비

In [1]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')

## Part 1. 도구 2개 등록과 이름 dispatch

완성 코드를 직접 쳐서 여러 도구를 등록하고 이름으로 골라 실행한다.

### 1-1. 도구 2개 정의

In [ ]:
# todo: 도구를 정의하고, 도구를 호출하는 예시를 작성하세요.
# todo: 운송장 번호로 택배 배송 상태를 반환하는 도구와, 두 수를 계산하는 도구를 정의하세요.

@tool
def calculate(a: float, b: float, opr: str) -> str:
    """두 개의 숫자 a,b를 받아서 op(+,-,*,/)로 계산한다."""

    # calculate(10, 20, '*') -> '30'
    table = {"+": a+b, "-": a-b, "*": a*b, "/" : a/b if b else None}

    return str(table.get(opr, "지원하지 않는 연산자 입니다."))

@tool
def get_delivery_status(tracking_number: str) -> str:
    """운송장 번호로 택배 배송 상태를 조회해서 반환한다."""
    return f"운송장 {tracking_number} : 배송중 (내일 도착 예정)"


### 1-2. 여러 도구 등록 + 이름→함수 표

`bind_tools`에 리스트로 넣고, 실행용 표를 만든다.

In [ ]:
# TODO: get_delivery_status·calculate를 tools 리스트로 만들고 bind_tools로 등록하세요
tool_list = [get_delivery_status, calculate]
llm_with_tools = llm.bind_tools(tool_list)

tool_map = {t.name: t for t in tool_list } #툴맵 함수표
tool_map

{'get_delivery_status': StructuredTool(name='get_delivery_status', description='운송장 번호로 택배 배송 상태를 조회해서 반환한다.', args_schema=<class 'langchain_core.utils.pydantic.get_delivery_status'>, func=<function get_delivery_status at 0x000001C0475A51C0>),
 'calculate': StructuredTool(name='calculate', description='두 개의 숫자 a,b를 받아서 op(+,-,*,/)로 계산한다.', args_schema=<class 'langchain_core.utils.pydantic.calculate'>, func=<function calculate at 0x000001C0475A5580>)}

### 1-3. 질문별 자동 선택

질문에 따라 AI가 다른 도구를 고른다.

In [ ]:
# todo: llm_with_tools를 호출한다
question = ["운송장 번호 CJ123456789 배송 상태가 궁금해요.", "3 곱하기 55는 얼마 입니까?"]

for q in question:
    tc = llm_with_tools.invoke(q).tool_calls[0]
    tc_result = tool_map[tc['name']].invoke(tc['args'])
    print(tc_result)

# tool_calls 에서 꺼내서
# tool_call 하기

운송장 CJ123456789 : 배송중 (내일 도착 예정)
165.0


### 1-4. 이름으로 골라 실행

요청된 이름으로 함수를 찾아 실행한다.

In [ ]:
# todo: "계산 툴"을 호출하는 예시를 작성하세요.


# TODO: tool_map에서 tc['name']에 해당하는 도구를 찾아 tc['args']로 실행하세요


### 1-4-보충. 결과 반환 → 최종 답

Day05는 `get_weather`로 전체 루프를 완성했다. 여기서는 이름으로 찾아 실행한 결과를 다시 모델에 돌려줘, 최종 답변까지 이어지는 것을 확인한다.

In [ ]:
# TODO: messages에 ai_msg와 ToolMessage(result, tool_call_id=tc["id"])를 순서대로 추가하고,
#       llm_with_tools.invoke(messages)로 최종 답을 받아 출력하세요




### 1-5. 오류 다뤄보기 — tool_map에 없는 이름이 오면?

`get_time` 도구는 등록만 하고 `tool_map`에는 넣지 않으면 어떻게 되는지 확인한다.

In [16]:
# TODO: get_time 도구를 정의하세요
from datetime import datetime

@tool
def get_time() -> str:
    """현재 시스템의 날짜와 시간을 반환한다."""
    now = datetime.now()
    period = "오전" if now.hour < 12 else "오후"
    hour_12 = now.hour % 12 or 12
    return f'지금은 {period} {hour_12}시 {now.minute}분 입니다.*^-^*'

# TODO: get_delivery_status·calculate·get_time 3개를 bind_tools로 등록하고,
#       small_tool_map에는 get_delivery_status·calculate 2개만 넣으세요
tool_list_1 = [get_delivery_status, calculate, get_time]

tool_map_1 = {t.name: t for t in tool_list_1 } #툴맵 함수표

llm_with_tools_3 = llm.bind_tools(tool_list_1)

# TODO: llm_incomplete를 호출하고, small_tool_map에서 tc['name']에 해당하는 도구를 찾아 tc['args']로 실행하세요

tc = llm_with_tools_3.invoke("지금 몇시야?").tool_calls[0]

try:
    result = tool_map_1[tc['name']].invoke(tc['args'])
    print(result)
except KeyError as e:
    print(type(e).__name__, e)


지금은 오전 10시 21분 입니다.*^-^*


### 1-6. 오류 다뤄보기 — 인자 이름이 틀리면?

AI가 뽑은 게 아니라, 우리가 직접 잘못된 인자 이름으로 도구를 호출하면 어떻게 되는지 확인한다.

In [20]:
# TODO: calculate에 op 대신 operator라는 잘못된 키로 호출해보세요
try:
    calculate.invoke({"a":3, "b":15, "operator":'*'})
except Exception as e:
    print(type(e).__name__)
    print(e)

ValidationError
1 validation error for calculate
opr
  Field required [type=missing, input_value={'a': 3, 'b': 15, 'operator': '*'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


## Part 2. 외부 API 래핑과 설명의 힘

외부 서비스를 도구로 감싸고, docstring 설명이 선택·인자 추출에 미치는 영향을 확인한다.

### 2-1. 외부 API를 도구로 (모의)

In [22]:
# todo: get_exchange_rate 도구를 정의하고, 통화 코드를 받아 환율을 반환하도록 docstring과 본문을 작성하세요 
# (예: USD, JPY, EUR 언급)
def get_exchange_rate(currency: str) -> str:
    """환율을 반환합니다."""
    rates = {"USD":1350, "JPY":8.83, "EUR": 1450}

    return f"{currency} 환율은 {rates.get(currency), '알 수 없음'} 원입니다."

# todo: get_exchange_rate를 tools 리스트에 추가하고 bind_tools로 등록하세요

llm_with_tools_1 = llm.bind_tools([get_exchange_rate])
llm_with_tools_1.invoke("달러 환율 얼마야?").tool_calls


[{'name': 'get_exchange_rate',
  'args': {'currency': 'USD'},
  'id': 'call_j8t5QEaEL5ydnX1i2oiJIZ65',
  'type': 'tool_call'}]

### 2-2. 설명의 힘

모호한 도구와 명확한 도구를 함께 등록하면, AI는 명확한 쪽을 고른다.

In [31]:
# todo: tool_a를 정의하고, 문자열을 받아 결과를 반환하도록 docstring과 본문을 작성하세요
def tool_a(info: str) -> str:
    """국가의 인구수를 반환합니다."""
    return f'입력: {info}값 입니다.'

# todo: tool_b 도구를 정의하고, 국가 이름을 받아 인구수를 반환하도록 docstring과 본문을 작성하세요
def tool_b(nation_name: str) -> str:
    """대상국가는 한국 등 아시아만 제공합니다.국가의 인구수를 반환합니다."""
    return f'{nation_name} 국가의 인구수는 약 5천 만명 입니다.'

# TODO: 국가의 인구수를 반환한다는 명확한 docstring을 작성하세요
# 대한민국 인구가 몇 명인가요?

llm_with_tools_2 = llm.bind_tools([tool_a, tool_b])
print(llm_with_tools_2.invoke("미국 인구가 몇 명인가요?").tool_calls)
print(llm_with_tools_2.invoke("대한민국 인구가 몇 명인가요?").tool_calls)

[{'name': 'tool_a', 'args': {'info': '미국'}, 'id': 'call_HN7OuDJCtFWvnXKVCfSiAxnG', 'type': 'tool_call'}]
[{'name': 'tool_b', 'args': {'nation_name': '대한민국'}, 'id': 'call_qFClYVsbrdJ1kFq4CZYcznle', 'type': 'tool_call'}]


## Part 2-확장. 다른 도메인에 적용하기 — 회사 리뷰 조회 도구

M2 Day01의 채용 공고 조회 도구에 이어, 회사 분위기를 알려주는 도구를 추가한다. 이번에도 모호한 버전과 명확한 버전을 함께 등록해 설명의 힘을 다시 확인한다.

### 2-3. 회사 리뷰 조회 도구 (모호 vs 명확)

In [ ]:
# todo: tool_b를 정의하고, 문자열을 받아 결과를 반환하도록 docstring과 본문을 작성하세요


# todo: get_company_review 도구를 정의하고, 회사 이름을 받아 재직자 리뷰 요약을 반환하도록 docstring과 본문을 작성하세요

# TODO: 회사의 재직자 리뷰 요약을 반환한다는 명확한 docstring을 작성하세요

# todo: get_company_review를 tools 리스트에 추가하고 bind_tools로 등록하세요

### 관찰 정리

- 환율 도구와 회사 리뷰 도구 모두에서, 모호한 도구 대신 명확한 도구가 선택됐는가?
- 두 도구(환율·인구, 리뷰·?)를 같이 등록했을 때, 이름이 비슷하면 AI가 헷갈릴 여지가 있을까?

> **참고:** 표현이 달라도 AI가 인자를 잘 뽑아내는지는 Day05에서 이미 확인했다(채용 공고 도구로) — 오늘은 같은 실습을 반복하지 않는다.

### 2-4. 이름이 비슷한 도구 두 개를 함께 등록하면?

관찰 정리에서 던진 질문을 직접 실험한다. `get_exchange_rate`(환율)와 역할이 비슷한 `get_gold_price`(금 시세)를 함께 등록하면, AI가 둘을 헷갈리지 않고 구분해 고르는지 확인한다.

In [33]:
# 1단계: unit(g 또는 don)에 따라 금 시세를 반환하는 도구 정의
@tool
def get_gold_price(unit: str) -> str:
    """금 시세를 원화로 반환한다. unit은 1그램은 g, 한 돈(3.75g)은 don을 사용한다."""
    prices = {"g": 100_000, "don": 375_000}
    unit = unit.lower()

    if unit not in prices:
        return f"{unit}은 지원하지 않는 단위입니다. g 또는 don을 사용하세요."

    unit_name = "1g" if unit == "g" else "한 돈(3.75g)"
    return f"금 {unit_name} 시세는 {prices[unit]:,}원입니다."


# 2단계: 환율 도구 목록에 금 시세 도구를 추가하고 모델에 등록
tools = [get_exchange_rate, get_gold_price]
llm_confuse = llm.bind_tools(tools)


# 3단계: 서로 다른 질문 3개로 모델이 선택한 도구와 인자를 확인
questions = [
    "오늘 달러 환율은 얼마야?",
    "금 1g 시세를 알려줘.",
    "금 한 돈 가격은 얼마야?",
]

for question in questions:
    ai_msg = llm_confuse.invoke(question)
    print(f"질문: {question}")
    print(f"tool_calls: {ai_msg.tool_calls}")
    print()


질문: 오늘 달러 환율은 얼마야?
tool_calls: [{'name': 'get_exchange_rate', 'args': {'currency': 'USD'}, 'id': 'call_nSN1M6bi7LAop5sLSULNNMV9', 'type': 'tool_call'}]

질문: 금 1g 시세를 알려줘.
tool_calls: [{'name': 'get_gold_price', 'args': {'unit': 'g'}, 'id': 'call_vyXEO2J7JUVYQTxPTnCZd4v6', 'type': 'tool_call'}]

질문: 금 한 돈 가격은 얼마야?
tool_calls: [{'name': 'get_gold_price', 'args': {'unit': 'don'}, 'id': 'call_yzzOftj1BDQEtjXrQtkxUM47', 'type': 'tool_call'}]



### 2-5. 관찰 정리 — 이름이 비슷해도 구분하는가?

- 환율·금 시세 둘 다 "원화 값을 알려주는" 도구인데도, description이 다르면 AI가 정확히 구분했는가?
- 만약 두 도구의 description까지 비슷했다면 어떻게 됐을까?

### Part 2-보충. 진짜 외부 API 연동해보기 — 모의가 아닌 진짜 호출

지금까지 등록한 도구(`get_exchange_rate`·`get_population`·`get_company_review`·`get_gold_price`)는 전부 dict에 값을 미리 박아 둔 **모의(mock)** 도구였다. 이번엔 실제로 네트워크를 타고 나가는 도구를 만든다 — 방식이 서로 다르다.

| 도구 | 방식 |
| --- | --- |
| `get_real_exchange_rate` | 금융 데이터 패키지(`yfinance`) 호출 |
| `find_starbucks_by_region` | 브라우저 개발자 도구로 잡은 REST API를 그대로 재현(POST) |
| `get_melon_chart` | HTML 페이지를 그대로 받아 표 구조를 파싱(스크래핑) |
| `get_naver_news` | HTML 페이지에서 본문 텍스트만 추출(스크래핑) |
| `get_naver_news_list` | 랭킹 페이지에서 기사 제목·링크 목록을 파싱(스크래핑) |


In [ ]:
# 환율 조회

import requests
import yfinance as yf
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
currency = "USD"
tickers = {"USD": "KRW=X", "JPY": "JPYKRW=X", "EUR": "EURKRW=X"}
ticker = tickers.get(currency)
if not ticker:
    print(f"{currency}는 지원하지 않는 통화입니다.")

price = yf.Ticker(ticker).info.get("regularMarketPrice")
price

In [18]:
import requests
import yfinance as yf
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

# TODO: get_real_exchange_rate(currency: str) 도구를 @tool로 정의하세요
#       - currency(USD/JPY/EUR)를 tickers 딕셔너리에서 찾고
#       - 없으면 "{currency}는 지원하지 않는 통화입니다."를 반환
#       - 있으면 yf.Ticker(ticker).info.get("regularMarketPrice")로 실시간 가격을 가져와
#         "{currency} 환율은 {price}원 (yfinance 실시간 조회)" 형태로 반환

@tool
def get_real_exchange_rate(currency: str)-> str:
    """currency 통화의 실제 환율을 실시간으로 조회한다."""
    tickers = {"USD": "KRW=X", "JPY": "JPYKRW=X", "EUR":"EURKRW=X"}
    ticker = tickers.get(currency)
    if not ticker:
        print(f"{currency}는 지원하지 않는 통화입니다.")

    price = yf.Ticker(ticker).info.get("regularMarketPrice")
    return f'{currency}의 환율은 {price}원 입니다. (yfinance 실시간 조회)'


In [10]:
# TODO: llm_real_fx = llm.bind_tools([get_real_exchange_rate])로 등록하고,
#       "엔화 환율 얼마야?"를 물어 tool_calls[0]을 실행해 결과를 출력하세요

llm_real_fx = llm.bind_tools([get_real_exchange_rate])
tc = llm_real_fx.invoke("엔화 환율 얼마야?").tool_calls[0]

In [11]:
tc['name'], tc['args']

('get_real_exchange_rate', {'currency': 'JPY'})

In [12]:
get_real_exchange_rate.invoke(tc['args'])

'JPY의 환율은 8.845원 입니다. (yfinance 실시간 조회)'

In [15]:
# 달러 환율도 확인해 보세요.
tc = llm_real_fx.invoke("달러 환율 얼마지?").tool_calls[0]
get_real_exchange_rate.invoke(tc['args'])

'USD의 환율은 1382.51원 입니다. (yfinance 실시간 조회)'

In [19]:
# 유로 환율도 확인해 보세요.
tc = llm_real_fx.invoke("유로 환율 얼마지?").tool_calls[0]
get_real_exchange_rate.invoke(tc['args'])

'EUR의 환율은 1584.5원 입니다. (yfinance 실시간 조회)'

In [59]:
# 스타벅스 매장 조회 — 전체 지역
SIDO_CODES = {"서울": "01", "^^":"13", "대구": "03", "대전": "04", "부산": "05", "울산": "06", "인천": "07", "?08": "08", "?09": "09", "?10": "10"}
count = 10

for sido, sido_cd in SIDO_CODES.items():
    payload = {
        "in_biz_cds": "0", "in_scodes": "0", "ins_lat": "0", "ins_lng": "0",
        "search_text": "", "p_sido_cd": sido_cd, "p_gugun_cd": "", "in_distance": "0",
        "in_biz_cd": "", "isError": "true", "searchType": "C", "set_date": "",
        "all_store": "0", "T03": "0", "T01": "0", "T27": "0", "T12": "0", "T09": "0",
        "T30": "0", "T05": "0", "T22": "0", "T21": "0", "T36": "0", "T43": "0",
        "Z9999": "0", "T64": "0", "P02": "0", "P10": "0", "P50": "0", "P20": "0",
        "P60": "0", "P30": "0", "P70": "0", "P40": "0", "P80": "0", "whcroad_yn": "0",
        "P90": "0", "P01": "0", "new_bool": "0", "iend": str(count),
        "rndCod": "49CY4NAI8F",
    }
    r = requests.post("https://www.starbucks.co.kr/store/getStore.do?r=S2FBUEYEFC", data=payload)
    stores = r.json()["list"]
    print(f"=== {sido} ===")
    print("\n".join(f"{s['s_name']} ({s['addr']})" for s in stores[:count]))
    print()


=== 서울 ===
역삼아레나빌딩 (서울특별시 강남구 역삼동 721-13 아레나빌딩)
논현역사거리 (서울특별시 강남구 논현동 142-2 정일빌딩)
신사역성일빌딩 (서울특별시 강남구 논현동 18-4 성일빌딩)
국기원사거리 (서울특별시 강남구 역삼동 648-22 동찬빌딩)
대치재경빌딩 (서울특별시 강남구 대치동 599 대원빌딩)
봉은사역 (서울특별시 강남구 삼성동 108-6 JBK TOWER 빌딩)
압구정윤성빌딩 (서울특별시 강남구 신사동 592 윤성빌딩)
코엑스별마당 (서울특별시 강남구 삼성동 159 코엑스)
삼성역섬유센터R (서울특별시 강남구 대치동 944-31 한국섬유산업연합회)
압구정R (서울특별시 강남구 신사동 621-1)

=== ^^ ===
전북김제DT (전북특별자치도 김제시 검산동 895-86)
전북남원DT (전북특별자치도 남원시 쌍교동 58-6)
익산모현 (전북특별자치도 익산시 모현동1가 320)
익산영등DT (전북특별자치도 익산시 영등동 340-11)
익산영등 (전북특별자치도 익산시 영등동 149-1 CGV익산)
익산어양 (전북특별자치도 익산시 영등동 816-4)
익산부송 (전북특별자치도 익산시 부송동 1066-4)
익산원광대 (전북특별자치도 익산시 신동 762-17)
익산동산DT (전북특별자치도 익산시 동산동 604-9)
전주송천DT (전북특별자치도 전주시 덕진구 송천동2가 487, 488)

=== 대구 ===
대구영대병원역DT (대구광역시 남구 봉덕동 595-7)
대구앞산DT (대구광역시 남구 대명동 493-38, 493-37, 493-26)
영남대학교의료원영의관 (대구광역시 남구 대명동 317-1 영남대학교(병원), 영남이공대학교)
대구가톨릭대학교병원 (대구광역시 남구 대명동 3056-6 대구가톨릭의과대학병원)
영남대학교의료원 (대구광역시 남구 대명동 317-1 영남대학교(병원), 영남이공대학교)
대구교대DT (대구광역시 남구 대명동 2008-1)
대구앞산스카이타운 (대구광역시 남구 대명동 585-4)
계명대동산병원 (대구광역시 달서구 신

In [39]:
# TODO: find_starbucks_by_region(sido: str, count: int = 3) 도구를 @tool로 정의하세요
#       - SIDO_CODES에 없는 sido면 "{sido}는 지원하지 않는 지역입니다."를 반환
#       - 있으면 아래 payload에서 p_sido_cd·iend만 sido_cd·count로 바꿔
#         "https://www.starbucks.co.kr/store/getStore.do?r=S2FBUEYEFC"로 POST 요청하고,
#         응답 json()["list"]에서 s_name·addr를 count개까지 "{이름} ({주소})" 줄로 모아 반환

SIDO_CODES = {"서울": "01", "대구": "03", "대전": "04", "부산": "05"}

@tool
def find_starbucks_by_region(sido: str, count: int = 3) -> str:
    """지역(서울, 대구, 대전, 부산)의 스타벅스 매장을 조회한다."""
    sido_cd = SIDO_CODES.get(sido)

    if not sido_cd :
        return f"{sido}는 지원하지 않는 지역입니다."
    
    payload_template = {
        "in_biz_cds": "0", "in_scodes": "0", "ins_lat": "0", "ins_lng": "0",
        "search_text": "", "p_sido_cd": sido_cd, "p_gugun_cd": "", "in_distance": "0",
        "in_biz_cd": "", "isError": "true", "searchType": "C", "set_date": "",
        "all_store": "0", "T03": "0", "T01": "0", "T27": "0", "T12": "0", "T09": "0",
        "T30": "0", "T05": "0", "T22": "0", "T21": "0", "T36": "0", "T43": "0",
        "Z9999": "0", "T64": "0", "P02": "0", "P10": "0", "P50": "0", "P20": "0",
        "P60": "0", "P30": "0", "P70": "0", "P40": "0", "P80": "0", "whcroad_yn": "0",
        "P90": "0", "P01": "0", "new_bool": "0", "iend": str(count),
        "rndCod": "49CY4NAI8F",
    }

    r = requests.post("https://www.starbucks.co.kr/store/getStore.do?r=S2FBUEYEFC", data=payload_template)
    stores = r.json()["list"]
    return ("\n".join(f"{s['s_name']} ({s['addr']})" for s in stores[:count]))



In [64]:
# TODO: llm_sbux = llm.bind_tools([find_starbucks_by_region])로 등록하고,
#       "대구에 스타벅스 매장 5개 알려줘"를 물어 tool_calls[0]을 실행해 결과를 출력하세요
llm_sbux = llm.bind_tools([find_starbucks_by_region])
tc = llm_sbux.invoke("서울에 스타벅스 매장 10개 알려줘").tool_calls[0]
tc['name'], tc['args']

('find_starbucks_by_region', {'sido': '서울', 'count': 10})

In [65]:
print(find_starbucks_by_region.invoke(tc['args']))

역삼아레나빌딩 (서울특별시 강남구 역삼동 721-13 아레나빌딩)
논현역사거리 (서울특별시 강남구 논현동 142-2 정일빌딩)
신사역성일빌딩 (서울특별시 강남구 논현동 18-4 성일빌딩)
국기원사거리 (서울특별시 강남구 역삼동 648-22 동찬빌딩)
대치재경빌딩 (서울특별시 강남구 대치동 599 대원빌딩)
봉은사역 (서울특별시 강남구 삼성동 108-6 JBK TOWER 빌딩)
압구정윤성빌딩 (서울특별시 강남구 신사동 592 윤성빌딩)
코엑스별마당 (서울특별시 강남구 삼성동 159 코엑스)
삼성역섬유센터R (서울특별시 강남구 대치동 944-31 한국섬유산업연합회)
압구정R (서울특별시 강남구 신사동 621-1)


> **참고:** 이 API는 브라우저 개발자 도구(Network 탭)로 사이트가 실제로 보내는 요청을 그대로 복사해 재현한 것이다 — 공식 문서가 없는 **비공식 API**라 파라미터 중 무엇이 실제로 검색에 쓰이는지 코드만 봐서는 알 수 없다. 직접 실측해보니: 위·경도(`ins_lat`/`ins_lng`)를 서울 좌표로 바꿔도 `p_sido_cd`를 대구(`03`)로 고정하면 결과는 여전히 대구 매장만 나왔다 — 즉 **실제 검색 기준은 좌표가 아니라 `p_sido_cd`**였다. 그래서 이 도구는 좌표 대신 지역 코드로 설계했다. 이렇게 문서 없는 API를 재현할 때는 파라미터 이름만 보고 짐작하지 말고, 값을 하나씩 바꿔가며 실제로 무엇이 결과를 바꾸는지 직접 확인해야 한다.

In [67]:
count = 10

"""멜론 실시간 차트 상위 곡을 조회한다."""
r = requests.get("https://www.melon.com/chart/index.htm", headers=HEADERS, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")
rows = soup.select("tr.lst50, tr.lst100")[:count]
lines = []
for row in rows:
    rank = row.select_one(".rank")
    title = row.select_one(".rank01 a")
    artist = row.select_one(".rank02 a")
    lines.append(f"{rank.get_text(strip=True)}위 {title.get_text(strip=True)} - {artist.get_text(strip=True)}")
print("\n".join(lines))

1위 LOVE ATTACK - RESCENE (리센느)
2위 퇴사할게여 (Narr. 기안84) - 소연 (SOYEON)
3위 갑자기 - 아이오아이 (I.O.I)
4위 이 별로부터 - 아이유
5위 BiiiG - BIGBANG (빅뱅)
6위 REDRED - CORTIS (코르티스)
7위 Pretty Girl - RESCENE (리센느)
8위 Deja Vu - RESCENE (리센느)
9위 Dear my crazy soulmate - 아이유
10위 BAD - ATEEZ(에이티즈)


In [68]:
# Codex
import pandas as pd
import requests
from bs4 import BeautifulSoup


def scrape_melon_chart() -> pd.DataFrame:
    """멜론 TOP100의 순위, 곡명, 가수, 좋아요 수를 반환한다."""
    chart_url = "https://www.melon.com/chart/index.htm"
    like_url = "https://www.melon.com/commonlike/getSongLike.json"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/140.0.0.0 Safari/537.36"
        ),
        "Referer": "https://www.melon.com/",
    }

    with requests.Session() as session:
        session.headers.update(headers)

        response = session.get(chart_url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        songs = []
        for row in soup.select("tr.lst50, tr.lst100"):
            rank = row.select_one("span.rank")
            title = row.select_one("div.rank01 a")
            artists = row.select("div.rank02 > a")
            song_id = row.get("data-song-no")

            if not all((rank, title, artists, song_id)):
                continue

            songs.append({
                "곡 ID": song_id,
                "순위": int(rank.get_text(strip=True)),
                "곡명": title.get_text(strip=True),
                "가수": ", ".join(
                    artist.get_text(" ", strip=True).replace("\xa0", " ")
                    for artist in artists
                ),
            })

        if not songs:
            raise RuntimeError("차트 데이터를 찾지 못했습니다. 멜론 페이지 구조를 확인하세요.")

        # 차트 HTML의 좋아요 수는 0으로 초기화되므로 별도 JSON 응답에서 조회한다.
        like_response = session.get(
            like_url,
            params={"contsIds": ",".join(song["곡 ID"] for song in songs)},
            headers={"X-Requested-With": "XMLHttpRequest"},
            timeout=10,
        )
        like_response.raise_for_status()
        like_counts = {
            str(item["CONTSID"]): int(item["SUMMCNT"])
            for item in like_response.json().get("contsLike", [])
        }

    for song in songs:
        song["좋아요"] = like_counts.get(song["곡 ID"], 0)

    return (
        pd.DataFrame(songs)
        .drop(columns="곡 ID")
        .sort_values("순위")
        .reset_index(drop=True)
    )


melon_chart_df = scrape_melon_chart()
melon_chart_df


,순위,곡명,가수,좋아요
0,1,LOVE ATTACK,RESCENE (리센느),148803
1,2,퇴사할게여 (Narr. 기안84),소연 (SOYEON),17470
2,3,갑자기,아이오아이 (I.O.I),78641
3,4,이 별로부터,아이유,26900
4,5,BiiiG,BIGBANG (빅뱅),54931
...,...,...,...,...
95,96,Love Love Love (Feat. Yoong Jin Of Casker),에픽하이 (EPIK HIGH),129214
96,97,STYLE,Hearts2Hearts (하츠투하츠),66257
97,98,사랑은 봄비처럼...이별은 겨울비처럼...,임현정,75868
98,99,REBEL HEART,IVE (아이브),122625


In [69]:
# TODO: get_melon_chart(count: int = 3) 도구를 @tool로 정의하세요
#       - "https://www.melon.com/chart/index.htm"을 requests.get(headers=HEADERS)로 받고
#       - BeautifulSoup으로 파싱해 "tr.lst50, tr.lst100" 행을 count개까지 순회하며
#         .rank · .rank01 a · .rank02 a 텍스트로 "{순위}위 {제목} - {가수}" 줄을 만들어 반환

@tool
def get_melon_chart(count: int = 3):
    """멜론 실시간 차트 상위 곡을 조회한다."""
    r = requests.get("https://www.melon.com/chart/index.htm", headers=HEADERS, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")
    rows = soup.select("tr.lst50, tr.lst100")[:count]
    lines = []
    for row in rows:
        rank = row.select_one(".rank")
        title = row.select_one(".rank01 a")
        artist = row.select_one(".rank02 a")
        lines.append(f"{rank.get_text(strip=True)}위 {title.get_text(strip=True)} - {artist.get_text(strip=True)}")

    return "\n".join(lines)    

In [70]:
# TODO: llm_melon = llm.bind_tools([get_melon_chart])로 등록하고,
#       "요즘 멜론차트 1위 뭐야?"를 물어 tool_calls[0]을 실행해 결과를 출력하세요
llm_melon = llm.bind_tools([get_melon_chart])

In [71]:
tc = llm_melon.invoke("요즘 멜론차트 1위 뭐야?").tool_calls[0]
tc['name'], tc['args']

('get_melon_chart', {'count': 1})

In [73]:
print(get_melon_chart.invoke(tc['args']))

1위 LOVE ATTACK - RESCENE (리센느)


In [ ]:
ranking_r = requests.get("https://news.naver.com/main/ranking/popularDay.naver", headers=HEADERS, timeout=10)
ranking_soup = BeautifulSoup(ranking_r.text, "html.parser")
news_url = ranking_soup.select_one("a.list_title")["href"]
print("오늘의 인기 기사:", news_url)

In [ ]:
url = "https://n.news.naver.com/article/081/0003679423?sid=105"

"""네이버 뉴스 기사 URL에서 제목과 본문 일부를 가져온다."""
r = requests.get(url, headers=HEADERS, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")
title = soup.select_one("h2#title_area")
body = soup.select_one("article#dic_area")
title_text = title.get_text(strip=True) if title else "제목을 찾을 수 없음"
body_text = body.get_text(strip=True)[:150] if body else ""
print(f"제목: {title_text}\n본문: {body_text}...")

In [ ]:
# TODO: get_naver_news(url: str) 도구를 @tool로 정의하세요
#       - url을 requests.get(headers=HEADERS)로 받고 BeautifulSoup으로 파싱해
#         "h2#title_area"(제목)·"article#dic_area"(본문)를 뽑아
#         "제목: ...\n본문: ..." 형태로 반환하세요(본문은 150자까지만)



llm_news = llm.bind_tools([get_naver_news])

# 특정 기사 URL을 하드코딩하면 나중에 기사가 내려가 링크가 죽을 수 있다 —
# 네이버 뉴스 랭킹에서 오늘의 인기 기사 링크를 매번 새로 가져온다
ranking_r = requests.get("https://news.naver.com/main/ranking/popularDay.naver", headers=HEADERS, timeout=10)
ranking_soup = BeautifulSoup(ranking_r.text, "html.parser")
news_url = ranking_soup.select_one("a.list_title")["href"]
print("오늘의 인기 기사:", news_url)

# TODO: llm_news를 호출해 f"{news_url} 이 기사 요약해줘"의 tool_calls[0]을 실행하고 결과를 출력하세요


In [ ]:
# TODO: get_naver_news_list(count: int = 5) 도구를 @tool로 정의하세요
#       - 네이버 뉴스 랭킹 페이지("https://news.naver.com/main/ranking/popularDay.naver")를 requests.get(headers=HEADERS)로 받고
#       - "a.list_title" 링크를 count개까지 "{번호}. {제목} ({링크})" 형태로 모아 반환



# TODO: llm_news_list = llm.bind_tools([get_naver_news_list])로 등록하고,
#       "오늘 인기 뉴스 3개만 보여줘"를 물어 tool_calls[0]을 실행해 결과를 출력하세요


### 진짜 도구 5개를 한 번에 등록 — 질문마다 알맞은 도구를 고르는가?

이름 dispatch 방식으로, 방식이 전혀 다른 도구 5개를 함께 등록해도 AI가 질문에 맞춰 올바른 도구를 고르는지 확인한다.

In [ ]:
# TODO: real_tools = [get_real_exchange_rate, find_starbucks_by_region, get_melon_chart, get_naver_news, get_naver_news_list]로
#       llm_real을 bind_tools로 만들고, real_tool_map(이름→함수 표)을 만드세요


questions = [
    "엔화 환율 얼마야?",
    "대구에 스타벅스 어디 있어?",
    "요즘 멜론차트 1위 뭐야?",
    f"{news_url} 이 기사 요약해줘",
    "오늘 인기 뉴스 3개만 보여줘",
]

# TODO: questions를 순회하며 llm_real.invoke(q) → tool_calls[0]을 real_tool_map으로 실행하고
#       질문·도구 이름·인자·결과를 출력하세요


> **참고:** 실제로 실행하면 5개 질문 모두 정확한 도구로 연결된다 — "엔화 환율"→`get_real_exchange_rate`(`{'currency': 'JPY'}`), "대구 스타벅스"→`find_starbucks_by_region`(`{'sido': '대구'}`), "멜론차트 1위"→`get_melon_chart`(`{'count': 1}`), 뉴스 URL을 포함한 질문→`get_naver_news`(URL만 정확히 뽑아냄), "오늘 인기 뉴스 3개만"→`get_naver_news_list`(`{'count': 3}`, 기사 제목·링크 3개를 실제로 가져옴). 모의 도구를 다룰 때와 dispatch 코드는 **완전히 동일**하다 — `@tool`로 감싸고 `bind_tools`에 등록하면, 내부가 dict 조회든 실제 네트워크 호출이든 LangChain·모델 입장에서는 차이가 없다.
>
> **모의 도구와 다른, 진짜 API/스크래핑만의 위험**도 있다: ① 사이트 구조가 바뀌면 `soup.select(...)`의 CSS 선택자가 깨진다(모의 도구는 절대 안 깨짐). ② 스타벅스처럼 공식 문서가 없는 비공식 API는 언제든 막히거나 파라미터가 바뀔 수 있다. ③ 짧은 시간에 반복 호출하면 차단(rate limit)될 수 있어, 실무에서는 캐싱이나 호출 간격 조절이 필요하다. ④ 이번 실습은 개인 학습 목적의 단발성 호출이지만, 실서비스에 쓰려면 각 사이트의 이용약관·크롤링 정책을 먼저 확인해야 한다.

### Part 2-보충-2. 지원하지 않는 값을 물으면?

Day05의 `get_job_requirements`(mock)는 존재하지 않는 회사를 물어도 검증 없이 아무 답이나 만들어냈다 — 그래서 `get_job_requirements_v2`로 회사 목록을 직접 검증해야 했다. 오늘의 진짜 API 도구들은 애초에 지원 범위가 정해져 있어(환율 3종, 지역 4곳), 그 범위를 벗어나면 어떻게 되는지 실제로 확인한다.

In [ ]:
# TODO: ["위안화 환율 알려줘", "제주도에 스타벅스 어디 있어?"]를 순회하며
#       llm_real로 tool_calls[0]을 확인하고, real_tool_map으로 실행해 결과를 출력하세요



> **참고:** 두 도구 모두 예외를 던지지 않고 "CNY는 지원하지 않는 통화입니다." / "제주는 지원하지 않는 지역입니다."를 그대로 반환한다면, 함수 안의 `if not ticker/sido_cd: return ...` 방어 로직 덕분이다. Day05의 `get_job_requirements_v2`가 **화이트리스트를 나중에 따로 추가**해야 했던 것과 달리, 오늘 도구들은 **애초에 지원 범위(딕셔너리 키)가 곧 검증 로직**이라 별도 버전을 만들 필요가 없었다.

## Part 3. 미니 프로젝트 — 반복 호출을 함수로 묶기

질문마다 알맞은 도구를 고르는 정확도는 Part 1·2에서 이미 충분히 확인했다. 이번엔 그 요청→실행→반환 반복을 **재사용 가능한 함수 하나**로 묶는다.

### 3-1. 도구 모음 + 여러 도구용 루프

In [ ]:
# todo: calculate·get_delivery_status를 tools_all 리스트로 만들고 bind_tools로 llm_multi를 등록하세요
# TODO: tool_map_all(이름→함수 표)을 만드세요


# todo: run_with_tools 함수를 작성하세요 — 질문을 받아 llm_multi를 호출하고,
#       tool_calls가 없으면 바로 ai_msg.content를 반환하세요

# TODO: ai_msg.tool_calls를 순회하며 tool_map_all로 실행하고 ToolMessage를 추가한 뒤,
#       llm_multi.invoke(messages).content를 반환하세요


### 3-2. 여러 질문에 적용

In [ ]:
# todo: 질문을 2개 만들어(도구가 필요한 것 1개, 필요 없는 것 1개) run_with_tools를 호출하고, 결과를 확인하세요



## Part 3-확장. M2 통합 프로젝트 — 면접 코치를 도구 모음으로 확장하기

M2 Day01의 `coach_with_tool_call`은 도구 1개(`get_job_requirements`)를 하드코딩했다. 오늘 배운 `tool_map` dispatch로 일반화해 `get_company_review`까지 함께 쓰게 한다.

### 방어 로직 + 도구 모음 (M1 Day04 · M2 Day01 재사용)

In [ ]:
# TODO: Day04(M1)의 위험 문구·금지어 목록과 input_guard·output_guard를 옮겨오세요




# todo: get_job_requirements 도구를 정의하고, 회사·직무를 받아 채용 공고 요건을 반환하도록 docstring과 본문을 작성하세요 (예: 3년 이상 경력, Python·SQL 우대, 팀 협업 경험 필수 언급)



# TODO: get_job_requirements·get_company_review를 담은 coach_tools 리스트를 만들고,
#       bind_tools로 coach_llm을 만들고, 이름→함수 표 coach_tool_map을 만드세요




### 도구 모음을 쓰는 면접 코치 함수

In [ ]:
# todo: coach_with_tools_v2 함수를 작성하고, 질문을 받아 coach_llm을 호출하고, tool_calls를 확인하세요


# TODO: tool_calls가 있으면 coach_tool_map으로 전부 실행 후 최종 답을,
#       없으면 ai_msg.content를 final에 담으세요



### 테스트 — 도구 2종·인젝션 시도

In [ ]:
# todo: 질문을 2개를 만들어 coach_with_tools_v2를 호출하고, 결과를 확인하세요


# TODO: 판정을 조작하려는 인젝션 질문을 넣어보세요




### 복합 질문 — 도구 두 개를 한 번에 요청하면?

한 질문에 두 도구가 모두 필요하면 `tool_calls`에 몇 개가 담기는지 먼저 직접 확인한 뒤, 전체 함수로도 실행해 본다.

In [ ]:
# TODO: coach_llm으로 "카카오 백엔드 개발자 채용 요건이랑 회사 분위기 둘 다 알려주세요."를 호출하고
#       tool_calls 개수와 각 도구 이름·인자를 출력하세요



In [ ]:
# TODO: coach_with_tools_v2로 같은 질문을 실행하세요



> **참고:** `coach_with_tools_v2`의 `for tc in ai_msg.tool_calls:` 반복문은 도구가 몇 개 요청되든 그대로 처리한다 — 도구 1개짜리로 짠 코드였지만 여러 개를 한 번에 처리하도록 이미 일반화돼 있었다.

### M2 Day02 정리

**확인 질문**
- `coach_with_tools_v2`가 도구 1개였던 M2 Day01 버전과 다른 점은 무엇인가?
- 도구를 하나 더 추가한다면(예: 연봉 정보 조회) 코드에서 무엇을 바꿔야 하는가?

## 확인 문제

1. `@tool`은 무엇으로 도구 설명을 만드는가?
2. 도구가 여러 개일 때, 어느 함수를 실행할지는 무엇을 보고 정하는가?
3. 1-5·1-6에서 확인한 것처럼, tool_map에 없는 이름이나 잘못된 인자 이름을 쓰면 무엇이 문제가 되는가?
4. docstring이 부실하면 어떤 문제가 생기는가?
5. Part 2와 Part 2-확장에서, 도구가 환율에서 회사 리뷰로 바뀌어도 변하지 않았던 것은 무엇인가?
6. Part 3-확장의 `coach_with_tools_v2`는 M2 Day01 버전과 비교해 어떤 부분이 일반화됐는가?

7. 이름이 비슷한 도구 두 개를 함께 등록해도 AI는 정확히 구분해 골랐는가?
8. 한 질문에 도구가 2개 필요할 때, `tool_calls`에는 몇 개가 담기는가?
9. Part 2-보충-2에서 지원하지 않는 지역·통화를 물었을 때, 도구는 왜 예외 없이 답을 만들어낼 수 있었는가? Day05의 `get_job_requirements_v2`와 무엇이 다른가?

## 정리·회고

오늘 배운 것을 3줄로 정리해 본다.

1. 도구가 여러 개일 때 실행할 함수를 고르는 방법(이름 dispatch)은 무엇이었는가?
2. 이름이 비슷한 도구 두 개(2-4~2-5)에서 무엇을 관찰했는가?
3. 한 질문에 도구가 2개 필요할 때 어떻게 처리되는지 확인했는가?

작성한 요약과 오늘 코드를 커밋한다.